# Secuenciador MIDI. De la notación simbólica a la interpretación

- Un archivo MIDI es esencialmente una partitura, una representación simbólica para la música. NO es sonido en sí.

- Más precisamente, es una *secuencia de eventos en el tiempo*:
  
  - *noteOn* con *nota* (0-127), el *canal* y *velocity* (velocidad de pulsación)
  
  - *noteOff* con la nota a apagar

  - La *duración* de la nota no está como tal!

  - Hay más tipos de eventos: control, exclusivos de sistema (sysex)... no nos interesan por ahora

- Utilizamos la librería *mido.py* para leer archivos midi
  
  - Hay otras librerías más sofisticadas en Python para procesamiento de midi, como **music21**...


In [ ]:
import mido

# lectura de archivo MIDI
events = mido.MidiFile('media/pirates.mid')

# para ver todos los eventos tal cual
for m in events: 
    print(m)


In [ ]:

# filtramos solo los noteOn/Off 
for m in events:    
    if m.type=='note_on':
        t = m.time
        # cuidado! hay midis que utilizan velocity=0 para el noteOff
        if m.velocity==0: print(f'{t:.4f} noteOff {m.note}  ch: {m.channel}')
        else: print(f'{t:.4f} noteOn  {m.note}  ch: {m.channel}  vel: {m.velocity} ')
    elif m.type=='note_off':
        print(f'{t:.4f} noteOff {m.note}  ch: {m.channel}')        

#### **Ojo**: los tiempos *deltaTime* representan tiempo transcurrido desde el último evento, no tiempo absoluto
... pero es muy fácil ir llevando un acumulado para obtener el tiempo relativo al inicio de la reproducción de la secuencia.



# Enganchando *el bucle* de renderizado/secuenciación

- Tal como estamos trabajando *TkInter* tiene el control de ejecución desde la llamada a *tk.mainloop()*

- Para implementar el bucle de secuenciación **nos colamos en la hebra** de *TkInter* con el método *after*
    
     ```after(time, func)```: después de *time* milisegundos llama a la función *func*

Haremos un método recursivo ```playLoop(item,time)```

- *item* es el siguiente elemento de la secuencia que tocará procesar (primero de una cola)

- *time* es el tiempo transcurrido desde el inicio de la reproducción, tiempo acumulado

- en cada llamada se actualiza *time* e *ítem*, si toca

- la llamada recursiva se deja programada con un *tick* de delay predefinido
    
    ```after(tick, lambda: playLoop(item,time))```


In [ ]:
%%writefile files/midiSequencerTk.py

import sys
sys.path.insert(0, "./files")        

import instrument
from tkinter import *
from consts import *
import sounddevice as sd
import numpy as np
import mido
import time 

# Secuenciador MIDI con un solo instrumento (el que se le pase o uno por defecto)

class MidiSequencerTk:
    # análogo a lo anterior
    def __init__(self,tk,instrument=None):
        # si no se pasa un instrumento, se crea uno por defecto
        if instrument == None:            
            self.instrument = instrument.Instrument(tk,amp=0.2,ratio=3,beta=0.6)
        else:
            self.instrument = instrument

        # Título de la venta
        frame = LabelFrame(tk, text="Midi Sequencer", bg="#908060")
        frame.pack(side=TOP)

        # Selector de archivo MIDI
        frameFile = Frame(frame, highlightbackground="blue", highlightthickness=6)
        frameFile.pack(side=TOP)
        Label(frameFile,text='Archivo MIDI: ').pack(side=LEFT)
 
        self.file = Entry(frameFile) #.pack(side=RIGHT)
        self.file.insert(14,"media/pirates.mid")
        self.file.pack(side=LEFT)

    
        # Ventana para información de eventos MIDI
        self.text = Text(frame,height=6,width=23)
        self.text.pack(side=RIGHT)

        # Botones de control play/stop
        playBut = Button(frame,text="Play", command=self.play)
        playBut.pack(side=TOP)
        stopBut = Button(frame,text="Stop", command=self.stop)
        stopBut.pack(side=BOTTOM)

        # para transponer la partitura, si se quiere
        self.transport = 0

        # tiempo entre eventos MIDI (en ms) para el loop de reproducción: "precision del reloj de secuenciación"
        # puede utilizarse para alterar la velocidad de reproducción (tempo) escalando este valor
        self.tick = 1 

        # y estado del secuenciador
        self.state = 'off'
        
    # obtención de la secuencia midi (noteOn/Off) con tiempos acumulados, relativos al inicio
    def getSeq(self,midiEvents):
        seq = []
        accTime = 0
        for m in midiEvents:
            accTime += m.time
            if m.type=='note_on':
                if m.velocity==0: seq.append((accTime,'noteOff',m.note+self.transport,m.channel))
                else: seq.append((accTime,'noteOn',m.note+self.transport,m.channel))    
            elif m.type=='note_off':
                seq.append((accTime,'noteOff',m.note+self.transport,m.channel))
        return seq

  
    # reproducción de la secuencia MIDI: prepara la secuencia y lanza el loop de reproducción playLoop
    def play(self):
        events = mido.MidiFile(self.file.get())
        seq = self.getSeq(events)
        print(seq)

        self.state = 'on'
        self.playLoop(seq)

    # método principal de reproducción: se llama a sí mismo cada tick ms para procesar los eventos MIDI que correspondan al tiempo acumulado accTime
    def playLoop(self,seq,item=0,accTime=0):   
        # final de la secuencia -> fin de la recursión
        if item>=len(seq) or self.state =='off':
            return

        # hay que procesar TODOS los ítems cuyo tiempo supere el crono accTime    
        while item<len(seq) and accTime>=seq[item][0]:
            (_,msg,midiNote,_chan) = seq[item]  # (time,'noteOff',midNote,channel)

            # mostramos el evento MIDI en la ventana de texto
            self.text.insert('6.0',  f'{msg} {midiNote}\n') 

            # activamos/apagamos nota
            if msg=='noteOn':  
                self.instrument.noteOn(midiNote)                   
            else: # msg noteOff    
                self.instrument.noteOff(midiNote)                   

            # y avanzmos ítem
            item += 1 


        # avanzammos crono con un factor de escalado
        accTime += self.tick/1000

        self.text.after(self.tick,lambda: self.playLoop(seq,item,accTime)) 

         
    def stop(self):
        self.instrument.stop()
        self.state = 'off'   


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "./files")        


from midiSequencerTk import *
import os    
from instrument import *


def test():
    def callback(outdata, frames, time, status):    
        if status: print(status)    
        #print(inputs)
        s = np.sum([i.next() for i in inputs],axis=0)
        s = np.float32(s)
        outdata[:] = s.reshape(-1, 1)

    os.system('xset r off')
    tk = Tk()
    ins = Instrument(tk)

    seq = MidiSequencerTk(tk,ins)
    #print(seq.seq)
    inputs = [ins]

    stream = sd.OutputStream(samplerate=SRATE, channels=1, blocksize=CHUNK, callback=callback)
    stream.start()

    tk.mainloop()

    stream.close()
    os.system('xset r on')

test()



# Arquitectura

![dibujo.png](media/dibujo.png)

# Inserciones (de efectos)

Podemos insertar un efecto "interceptando" cualquier envío de señal de un generador (líneas rosas)

- El efecto toma la señal de entrada del generador con **next()**

- Y devuelve la señal transformada implementado un **next()**

Vamos probar con un filtro IIR sencillo. 

- Incluimos la frecuencia de corte como parámetro en un Slider


In [ ]:
%%writefile lpFilter.py

import sys
sys.path.insert(0, "./files")        


from consts import *
import numpy as np
from tkinter import *
from slider import *
#from controller import *

class LPfilter:
    def __init__(self,tk,signal=None,freq=10000):        
        # señal de entrada
        self.signal = signal

        self.memo = 0.0
        frame = LabelFrame(tk, text="LP Filter", bg="#908060")
        frame.pack(side=TOP)

        self.freqS = Slider(frame,'freq',packSide=TOP,
                           ini=freq,from_=50,to=20000,step=100) 
        
    def next(self):    
        print('entro')  
        # calculo de alpha en función de la frecuencia de corte       
        alpha = np.exp(-2*np.pi*self.freqS.get() / SRATE)       

        # chunk de entrada pedido next() al generador 
        bloque= self.signal.next()
        bloque[0] = alpha * self.memo + (1-alpha) * bloque[0]
        for i in range(1,CHUNK):         
            bloque[i] = alpha * bloque[i-1] + (1-alpha) * bloque[i]            
        
        self.memo = bloque[CHUNK-1]
        return bloque


Vamos a conectar el filtro a la salida del instrumento:
- Probar con distintas frecuencias de corte utilizando el teclado 
- Probar con el secuencidor midi

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "./files")        


from consts import *
from midiSequencerTk import *
import os    
from instrument import *
from lpFilter import *

def test():
    def callback(outdata, frames, time, status):    
        if status: print(status)    
        #print(inputs)
        s = np.sum([i.next() for i in inputs],axis=0)
        s = np.float32(s)
        outdata[:] = s.reshape(-1, 1)

    os.system('xset r off')
    tk = Tk()

    ins = Instrument(tk)
    seq = MidiSequencerTk(tk,instrument=ins)
    filter = LPfilter(tk,ins)      
    inputs = [filter]

   

    stream = sd.OutputStream(samplerate=SRATE, channels=1, blocksize=CHUNK, callback=callback)
    stream.start()

    tk.mainloop()

    stream.close()

    os.system('xset r on')


test()


# Añadiendo efectos con librería externa: PedalBoard

...

In [ ]:
%%writefile reverb.py

from consts import *
from pedalboard import Pedalboard, Compressor, Reverb
from slider import *


class MiReverb:
    def __init__(self,tk,signal):
        self.signal = signal
        frame = LabelFrame(tk, text="PB Reverb", bg="#808090")
        frame.pack(side=BOTTOM)
        
        self.wetlS = Slider(frame,'WetLevel',packSide=TOP,
                           ini=0.0,from_=0.0,to=1.0,step=0.05) 
        
        self.board = Pedalboard([Reverb()])
        self.rev = self.board[0]
        self.rev.wet_level = .3

    def next(self):
        chunk = self.signal.next()
        self.rev.wet_level = self.wetlS.get()
        #print(self.rev.wet_level)
        chunk = self.board.process(chunk,sample_rate=SRATE, reset=False)
        return chunk


In [ ]:
%%writefile effects.py

from consts import *
from pedalboard import Pedalboard, Compressor, Reverb, Gain, Chorus, LadderFilter, Phaser
from slider import *


class Effects:
    def __init__(self,tk,signal):
        self.signal = signal
        frame = LabelFrame(tk, text="PB Reverb", bg="#808090")
        frame.pack(side=BOTTOM)
        
        self.wetlS = Slider(frame,'WetLevel',packSide=TOP,                          ini=0.0,from_=0.0,to=1.0,step=0.05) 
        
        self.board = Pedalboard([
            Compressor(threshold_db=-50, ratio=25),
            Gain(gain_db=30),
            Chorus(),
            LadderFilter(mode=LadderFilter.Mode.HPF12, cutoff_hz=900),
            Phaser(),
            #Convolution("./guitar_amp.wav", 1.0),
            Reverb(room_size=0.25),
        ])


    def next(self):
        chunk = self.signal.next()
        #self.rev.wet_level = self.wetlS.get()
        #print(self.rev.wet_level)
        chunk = self.board.process(chunk,sample_rate=SRATE, reset=False)
        return chunk

In [ ]:
from midiSequencerTk import *
import os    
from instrument import *
import effects


def test():
    def callback(outdata, frames, time, status):    
        if status: print(status)    
        #print(inputs)
        s = np.sum([i.next() for i in inputs],axis=0)
        s = np.float32(s)
        outdata[:] = s.reshape(-1, 1)

    os.system('xset r off')
    tk = Tk()
    ins = Instrument(tk)
    seq = MidiSequencerTk(tk,instrument=ins)
    
    #insRev = reverb.MiReverb(tk,ins)      
    # inputs = [insRev]
    
    insRev = effects.Effects(tk,ins)          
    inputs = [insRev]

    stream = sd.OutputStream(samplerate=SRATE, channels=1, blocksize=CHUNK, callback=callback)
    stream.start()

    tk.mainloop()

    stream.close()

    os.system('xset r on')


test()
